# Traditional vegetation segmentation baseline

This notebook tests unvalidated classical vegetation masks on the selected representative images. It does not create ground truth, quantify accuracy, train a model, or perform segmentation with a foundation model. A green pixel is not automatically the target plant.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src').is_dir() and (candidate / 'data' / 'raw' / 'first_sample').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook from the thesis-project folder or its notebooks folder.')

ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
RAW = ROOT / 'data' / 'raw' / 'first_sample'
RESULTS = ROOT / 'results' / 'traditional_segmentation'
RESULTS.mkdir(parents=True, exist_ok=True)


## Methods and consistent parameters

ExG is `2G − R − B`; HSV uses a documented hue/saturation range; LAB uses the green–red `a*` channel. RGB, HSV, and LAB are colour spaces, not segmentation methods. Each method uses one manual threshold setting across all representative images, plus an Otsu result for comparison. Otsu is not assumed to be appropriate: inspect its mask rather than treating it as an optimum.

In [ ]:
from src.segmentation import PARAMETERS
import pandas as pd
pd.DataFrame([{'Parameter': key, 'Value': value} for key, value in PARAMETERS.items()])

## Selected images

The same five manually selected images are used throughout. No parameter is tuned separately to make an individual image appear perfect.

In [ ]:
selection = pd.read_csv(ROOT / 'results' / 'representative_images.csv').dropna(subset=['Filename']).copy()
if selection.empty: raise ValueError('Select representative filenames in results/representative_images.csv first.')
selection

## Generate and inspect masks

Each saved figure contains: original image, manual-threshold raw mask, Otsu comparison mask, minimally cleaned manual mask, and overlay. Cleaning is only one 3×3 opening and closing. Inspect large leaves, thin stems/tendrils, background vegetation, trellis, blur, lighting, and whether the same parameters generalize.

In [ ]:
from src.image_analysis import load_rgb
from src.segmentation import exg_masks, hsv_masks, lab_masks, overlay_mask
import matplotlib.pyplot as plt

methods = {'exg': ('Excess Green', exg_masks), 'hsv': ('HSV vegetation', hsv_masks), 'lab': ('LAB a* vegetation', lab_masks)}
for _, record in selection.iterrows():
    path = RAW / record['Filename']
    rgb = load_rgb(path)
    for key, (name, method) in methods.items():
        raw_mask, otsu, cleaned = method(rgb)
        fig, axes = plt.subplots(1, 5, figsize=(20, 5))
        panels = [(rgb, 'Original', None), (raw_mask, 'Manual threshold mask', 'gray'), (otsu, 'Otsu comparison mask', 'gray'), (cleaned, 'Cleaned manual mask', 'gray'), (overlay_mask(rgb, cleaned), 'Cleaned-mask overlay', None)]
        for ax, (image, title, cmap) in zip(axes, panels): ax.imshow(image, cmap=cmap); ax.set_title(title); ax.axis('off')
        fig.suptitle(f"{record['Image_ID']} | {name}"); fig.tight_layout()
        fig.savefig(RESULTS / f"{path.stem}_{key}.png", dpi=180, bbox_inches='tight')
        plt.show(); plt.close(fig)

## Qualitative review

Fill `qualitative_summary.csv` after inspecting every method across every representative image. Use Good, Moderate, or Poor only as qualitative observations, not accuracy claims. Explicitly record failure cases. Do not calculate IoU, Dice, precision, or recall without validated reference masks.

In [ ]:
summary_path = RESULTS / 'qualitative_summary.csv'
if not summary_path.exists():
    pd.DataFrame({'Method':['ExG','HSV','LAB'], 'Large leaves':['','',''], 'Thin stems':['','',''], 'Background vegetation':['','',''], 'Trellis interference':['','',''], 'Blur robustness':['','',''], 'Lighting robustness':['','',''], 'Overall suitability':['','',''], 'Failure notes':['','','']}).to_csv(summary_path, index=False)
pd.read_csv(summary_path)

## Interpretation boundary

These masks are classical baselines, not validated plant masks. Likely failure modes to document include background vegetation incorrectly included as foreground, trellis/background edges affecting thresholding, blur reducing edge and colour reliability, and thin structures being removed by cleaning. A later SAM2 experiment is justified only if this visual baseline remains insufficient after documented, consistent parameters—not because it is a more popular method.